# Biomed Knowladge Grap


## Data preparation

#### General imports

In [8]:
%load_ext autoreload
%autoreload 2


import sys
from pathlib import Path
sys.path.append('..')
ROOT = Path.cwd().parent          # iz notebooks/ na root
sys.path.insert(0, str(ROOT))

from src import config, data, processing
import rich
import numpy as np


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Check connection to database

In [9]:
driver = data.get_driver()
driver.verify_connectivity()

def check_db_data():
    with driver.session() as s:
        nodes = s.run('MATCH (n) RETURN count(n) AS c').single()['c']
        rels = s.run('MATCH ()-[r]->() RETURN count(r) AS c').single()['c']

    print(f'nodes: {nodes}, relationships: {rels}')

In [10]:
check_db_data()

nodes: 15763, relationships: 160330


#### Load graph

In [ ]:
data.clear_db()
data.load_hetionet()



In [ ]:
check_db_data()


In [12]:
G, index = processing.prepare_graph()
rng = np.random.default_rng(config.SEED)
G_train, train, val, test = processing.split_edges(G, index, rng)

[load_from_db]: 15,763 nodes, 160,330 edges
[to_simple_undirected]: 15,763 nodes, 160,330 edges (0 self-loops, 0 parallel dropped)
[build_node_index]: 15,763 nodes (Disease 0 - 135, Gene 136 - 15762)
[split_edges] node_of: {0: ('Disease', 'DOID:0050156'), 1: ('Disease', 'DOID:0050425'), 2: ('Disease', 'DOID:0050741'), 3: ('Disease', 'DOID:0050742'), 4: ('Disease', 'DOID:0060073'), 5: ('Disease', 'DOID:0060119'), 6: ('Disease', 'DOID:10021'), 7: ('Disease', 'DOID:10153'), 8: ('Disease', 'DOID:1024'), 9: ('Disease', 'DOID:10283'), 10: ('Disease', 'DOID:10534'), 11: ('Disease', 'DOID:10608'), 12: ('Disease', 'DOID:10652'), 13: ('Disease', 'DOID:10763'), 14: ('Disease', 'DOID:10811'), 15: ('Disease', 'DOID:10871'), 16: ('Disease', 'DOID:1094'), 17: ('Disease', 'DOID:10941'), 18: ('Disease', 'DOID:10976'), 19: ('Disease', 'DOID:11054'), 20: ('Disease', 'DOID:11119'), 21: ('Disease', 'DOID:1115'), 22: ('Disease', 'DOID:11239'), 23: ('Disease', 'DOID:11476'), 24: ('Disease', 'DOID:11555'), 25

In [13]:
all_pos = np.vstack([train, val, test])
space = processing.candidate_space(all_pos, len(index))
forbidden = set(processing.encode(all_pos, space["n_nodes"]))

[negative_space]: 134 bolesti × 5,392 gena = 722,528 mogućih parova


In [14]:
train_neg = processing.sample_negatives(train, space, forbidden, config.NEGATIVE_RATIO_TRAIN, rng)
valid_neg = processing.sample_negatives(val, space, forbidden, config.NEGATIVE_RATIO_VALIDATION, rng)
test_neg  = processing.sample_negatives(test, space, forbidden, config.NEGATIVE_RATIO_VALIDATION, rng)

[sample_negatives]: 10,157 negatives (uniform, 1:1 on 10,157 positives)
[sample_negatives]: 12,330 negatives (uniform, 1:10 on 1,233 positives)
[sample_negatives]: 12,330 negatives (uniform, 1:10 on 1,233 positives)


#### Save processed data

In [ ]:
splits = {
    "sampler": config.NEGATIVE_SAMPLER,
    "G_train": G_train,
    "train_pos": train, "train_neg": train_neg,
    "val_pos": val,     "val_neg": valid_neg,
    "test_pos": test,   "test_neg": test_neg,
    "test_cold": processing.cold_start_mask(train, test),
}

processing.save_splits(G, index, splits)

#### Load processed data

Reads `data/processed/` only — the database is not touched.

In [ ]:
G, index, splits = processing.load_splits()

G_train = splits["G_train"]
train, val, test = splits["train_pos"], splits["val_pos"], splits["test_pos"]
train_neg, valid_neg, test_neg = splits["train_neg"], splits["val_neg"], splits["test_neg"]

rich.print({name: value.shape for name, value in splits.items()
            if isinstance(value, np.ndarray)})